# 2. Compare covariance models

Separate PCA, joint PCA, and PLSSVD on the same recordings. These are descriptive, in-sample comparisons. Cluster computations are separate from notebook visualization.

In [ ]:
from pathlib import Path
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from IPython.display import display

# Select match, order, or random (all 20 random associations).
MODE = 'match'
UNCERTAINTY = 'SD'  # SD or SEM; random pairings share the same iEEG participants.
ROOT = Path(os.environ.get('MIB_PROJECT_ROOT', Path.cwd())).resolve()
if not (ROOT / 'vs0_run_analysis.py').is_file():
    ROOT = ROOT / 'iEEGvsMEG'
sys.path.insert(0, str(ROOT))
OUT_ROOT = Path(os.environ.get('MIB_OUTPUT_ROOT', ROOT / 'out'))
ANALYSIS_ROOT = OUT_ROOT / 'vs2_cov_models'
CONFIG = json.loads((ANALYSIS_ROOT / 'config.json').read_text())
N_COMPONENTS = CONFIG['parameters']['n_components']
MODEL_STYLES = {'separate_pca': '-', 'joint_pca': '--', 'plssvd': ':'}
MODEL_COLORS = {'separate_pca': '#E5C400', 'joint_pca': '#EA3B52', 'plssvd': '#E68613'}
from types import SimpleNamespace
from vs1_utils_coverage_figures import apply_style, MODALITY_COLORS
apply_style()
if MODE not in ('match', 'order', 'random') or UNCERTAINTY not in ('SD', 'SEM'):
    raise ValueError('Select match/order/random and SD/SEM.')
PAIRINGS = [f'random_{i:03d}' for i in range(1, 21)] if MODE == 'random' else [MODE]

## Load saved results

The notebook reads `out/vs2_cov_models/<pairing>/` without reloading recordings or fitting models. Random mode requires all 20 completed random associations. Bands describe variation across these associations; SEM is descriptive and does not imply independent participant samples.

For time courses and covariance displays, random components are matched to the match pairing by absolute temporal iEEG correlation, with one-to-one assignment and sign alignment. Both modality contributions retain the same component permutation; joint PCA and PLSSVD retain their relative signs. Separate PCA's independent MEG sign is aligned separately. Cumulative variance and within-model scalar metrics retain their saved component ranks. All matching and plotting code is contained below.

In [ ]:
def load_run(pairing):
    folder = ANALYSIS_ROOT / pairing
    manifest = json.loads((folder / 'manifest.json').read_text())
    if manifest.get('status') != 'complete' or manifest.get('pairing') != pairing:
        raise ValueError(f'Incomplete or incorrect result: {folder}')
    if manifest['configuration']['parameters'] != CONFIG['parameters']:
        raise ValueError(f'Configuration differs from saved result: {folder}')
    metadata = json.loads((folder / 'models.json').read_text())
    models = {}
    for name, values in metadata.items():
        # Gram matrices and spatial weights are unnecessary for these saved-result plots.
        with np.load(folder / f'{name}.npz', allow_pickle=False) as saved:
            models[name] = SimpleNamespace(name=name, conditions=values['conditions'],
                times=saved['times'].copy(), ieeg_scores=saved['ieeg_scores'].copy(),
                meg_scores=saved['meg_scores'].copy())
    return models

RUN_MODELS = {pairing: load_run(pairing) for pairing in PAIRINGS}
REFERENCE = load_run('match') if MODE == 'random' else RUN_MODELS[PAIRINGS[0]]
ALIGNMENT, ALIGNED, matching_rows = {}, {}, []
for pairing, models in RUN_MODELS.items():
    ALIGNED[pairing], ALIGNMENT[pairing] = {}, {}
    if set(models) != set(REFERENCE):
        raise ValueError('Model sets differ across pairings.')
    for name, model in models.items():
        ref = REFERENCE[name]
        if not np.array_equal(model.times, ref.times) or model.conditions != ref.conditions:
            raise ValueError('Time axes or condition order differ across pairings.')
        k = min(N_COMPONENTS, ref.ieeg_scores.shape[1], model.ieeg_scores.shape[1])
        if k != min(N_COMPONENTS, ref.ieeg_scores.shape[1]):
            raise ValueError('Insufficient components in a random pairing.')
        xref, x = ref.ieeg_scores[:, :k], model.ieeg_scores[:, :k]
        if MODE == 'random':
            r = np.corrcoef(xref.T, x.T)[:k, k:]
            if not np.isfinite(r).all():
                raise ValueError('Undefined component correlation; inspect constant components.')
            rows, permutation = linear_sum_assignment(-abs(r))
            signs = np.where(r[rows, permutation] < 0, -1., 1.)
        else:
            permutation, signs = np.arange(k), np.ones(k)
            r = np.eye(k)
        xs = model.ieeg_scores[:, permutation]*signs
        ys = model.meg_scores[:, permutation]*signs
        if MODE == 'random' and name == 'separate_pca':
            ry = np.array([np.corrcoef(ref.meg_scores[:, j], ys[:, j])[0, 1] for j in range(k)])
            ys *= np.where(ry < 0, -1., 1.)
        ALIGNED[pairing][name] = (xs, ys)
        ALIGNMENT[pairing][name] = {int(original+1): int(target+1) for target, original in enumerate(permutation)}
        for j, original in enumerate(permutation):
            matching_rows.append(dict(pairing=pairing, model=name, reference_component=j+1,
                                      original_component=int(original+1), abs_r=abs(r[j, original])))
MATCHING_TABLE = pd.DataFrame(matching_rows)
if MODE == 'random':
    display(MATCHING_TABLE)

def summarize(values):
    values = np.asarray(values, dtype=float)
    count = np.sum(np.isfinite(values), axis=0)
    mean = np.nanmean(values, axis=0)
    spread = np.nanstd(values, axis=0, ddof=1) if len(values) > 1 else np.zeros_like(mean)
    if UNCERTAINTY == 'SEM':
        spread = spread / np.sqrt(count)
    return mean, spread

def curve(ax, time, values, **style):
    mean, spread = summarize(values)
    ax.plot(time, mean, **style)
    if MODE == 'random':
        ax.fill_between(time, mean-spread, mean+spread, color=style.get('color'), alpha=.18)
    return mean, spread

print(f'{MODE}: {len(PAIRINGS)} pairing(s); uncertainty: {UNCERTAINTY}')
display(pd.json_normalize(CONFIG['parameters']).T.rename(columns={0: 'value'}))

## Fit the three models

| Model | Objective | Time courses used in matching |
|---|---|---|
| Separate PCA | Maximise each modality's own variance | `X Wx` and `Y Wy` from independent fits |
| Joint PCA | Maximise variance of `[X Y]` | Separate contributions `X Wx` and `Y Wy`; their sum is the joint score |
| PLSSVD | Maximise cross-covariance with unit-norm weights | `X Wx` and `Y Wy` from SVD of `X.T @ Y` |

All matrices are centred across observations. 

The helper uses exact sample-space decompositions, avoiding a huge source × electrode cross-covariance matrix for full MEG concatenation. Joint PCA is compared using its separate modality contributions, never by correlating its common score with itself.

In [ ]:
print(CONFIG['command'])
print('Saved model files are used; this notebook does not run the analysis.')

## Variance explained and shared cross-covariance

`evaluate_cov_models` reports, cumulatively for the first **k** components:

- **`ieeg_variance_explained` / `meg_variance_explained`:** `1 − reconstruction error / total centred signal variance`, with a separate denominator for each modality. Separate PCA and PLS reconstruct through their own weights. Joint PCA reconstructs each modality using the joint score and its corresponding weight block.
- **`shared_crosscov_fraction`:** the fraction of squared cross-covariance captured by the retained feature subspaces. It is `||Qx.T C Qy||² / ||C||²`, where `C = X.T Y/(n−1)` and Qx/Qy are orthonormal bases of the retained weight spaces. For PLSSVD this equals the retained squared singular values divided by all squared singular values.

The last measure is **cross-covariance energy, not a uniquely identified shared biological variance**. It must not be added to the two variance-explained fractions. It depends on preprocessing and can include shared stimulus timing and sampling noise.

Native score variances and paired cross-covariances are included as additional columns. These reproduce the decomposition used in `LB_Summary.ipynb`, with explicit centring and denominators.

Separate PCA is expected to maximise each modality’s reconstruction variance, and PLSSVD to maximise retained cross-covariance energy. Those advantages are consequences of their objectives, not independent evidence of a better model.

In [ ]:
FIGSIZE = (15, 4)
variance = pd.concat([pd.read_csv(ANALYSIS_ROOT / pairing / 'variance.csv').assign(pairing=pairing)
                      for pairing in PAIRINGS], ignore_index=True)
fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, constrained_layout=True)
metrics = ['ieeg_variance_explained', 'meg_variance_explained', 'shared_crosscov_fraction']
for ax, metric in zip(axes, metrics):
    for name in REFERENCE:
        sub = variance[variance.model == name]
        pivot = sub.pivot(index='pairing', columns='k', values=metric).reindex(PAIRINGS)
        curve(ax, pivot.columns.to_numpy(), pivot.to_numpy(), linestyle=MODEL_STYLES[name], color=MODEL_COLORS[name], label=name)
    ax.set(xlabel='Number of components', ylabel='Fraction', title=metric, ylim=(0, 1.02)); ax.grid(True)
if MODE == 'random': fig.suptitle(f'Mean ± {UNCERTAINTY} · {len(PAIRINGS)} random pairings')
axes[-1].legend()
plt.show()

## Time courses

In [ ]:
PLOT_COMPONENTS = N_COMPONENTS
STANDARDIZE_SCORES = True
for name, model in REFERENCE.items():
    k = min(PLOT_COMPONENTS, ALIGNED[PAIRINGS[0]][name][0].shape[1])
    fig, axes = plt.subplots(k, 1, figsize=(12, 2.5*k), squeeze=False, constrained_layout=True)
    fig.suptitle(name.replace('_', ' ') + (f' · mean ± {UNCERTAINTY} · {len(PAIRINGS)} pairings' if MODE == 'random' else ''))
    for pc, ax in enumerate(axes[:, 0]):
        for m, modality in enumerate(('iEEG', 'MEG')):
            traces = []
            for pairing in PAIRINGS:
                trace = ALIGNED[pairing][name][m][:, pc].copy()
                if STANDARDIZE_SCORES:
                    sd = trace.std(); trace = (trace-trace.mean())/(sd if sd > 0 else 1.)
                traces.append(trace.reshape(-1, len(model.times)))
            traces = np.asarray(traces)
            for condition in range(traces.shape[1]):
                curve(ax, model.times, traces[:, condition], color=MODALITY_COLORS[modality],
                      linestyle=['-', '--', ':', '-.'][condition % 4], **({'label': modality} if pc == 0 else {}))
            if pc == 0: ax.legend()
        ax.set(title=f'PC {pc+1}', xlabel='Time (s)', ylabel='Standardized score' if STANDARDIZE_SCORES else 'Score'); ax.grid(True)
    plt.show()

## Covariance structure

Heatmaps retain signed covariance. The componentwise line plot shows score variance in each modality and **2 |Cov(iEEG, MEG)|**. These three quantities do not in general sum to the variance of the summed scores.

In [ ]:
PLOT_COMPONENTS = N_COMPONENTS
COMMON_SCALE = False
FIGSIZE = (15, 4)
COVARIANCES = {}
for name, model in REFERENCE.items():
    k = min(PLOT_COMPONENTS, ALIGNED[PAIRINGS[0]][name][0].shape[1])
    stack = []
    for pairing in PAIRINGS:
        x, y = (v[:, :k] for v in ALIGNED[pairing][name])
        x, y = x-x.mean(0), y-y.mean(0)
        stack.append([x.T@x/(len(x)-1), y.T@y/(len(x)-1), x.T@y/(len(x)-1)])
    COVARIANCES[name] = np.asarray(stack)
    matrices, spreads = summarize(stack)
    limit = max(np.max(abs(matrix)) for matrix in matrices) or 1.
    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, constrained_layout=True)
    for ax, matrix, spread, title in zip(axes, matrices, spreads, ['iEEG covariance', 'MEG covariance', 'Signed cross-covariance']):
        bound = limit if COMMON_SCALE else (np.max(abs(matrix)) or 1.)
        im = ax.imshow(matrix, cmap='RdBu_r', vmin=-bound, vmax=bound)
        ax.set(title=title, xticks=np.arange(k), xticklabels=np.arange(1,k+1), yticks=np.arange(k), yticklabels=np.arange(1,k+1)); ax.grid(True)
        if MODE == 'random':
            for i in range(k):
                for j in range(k):
                    ax.text(j, i, f'{matrix[i,j]:.2g}\n±{spread[i,j]:.1g}', ha='center', va='center', fontsize=6)
        fig.colorbar(im, ax=ax)
    fig.suptitle(name + (f' · mean ± {UNCERTAINTY}' if MODE == 'random' else '')); plt.show()

In [ ]:
PLOT_COMPONENTS = N_COMPONENTS
FIGSIZE = (9, 4)
for name in REFERENCE:
    covariance = COVARIANCES[name]
    k = min(PLOT_COMPONENTS, covariance.shape[-1])
    components = np.arange(1, k+1)
    fig, ax = plt.subplots(figsize=FIGSIZE, constrained_layout=True)
    for index, color, label in [(0, MODALITY_COLORS['iEEG'], 'Var(iEEG score)'),
                               (1, MODALITY_COLORS['MEG'], 'Var(MEG score)'),
                               (2, '#555555', '2 |Cov(iEEG, MEG)|')]:
        values = np.diagonal(covariance[:, index, :k, :k], axis1=1, axis2=2)
        if index == 2: values = 2*abs(values)
        curve(ax, components, values, marker='o', linestyle='-', color=color, label=label)
    ax.set(title=name.replace('_', ' '), xlabel='Component', ylabel='Score variance / covariance magnitude'); ax.legend(); ax.grid(True)
    if MODE == 'random': ax.set_title(name.replace('_', ' ') + f' · mean ± {UNCERTAINTY}')
    plt.show()

## Within-modality model similarity
**Subspace overlap** compares all retained patterns together and permits rotations/mixing of their axes. **Mean matched |r|** compares individual component pairs and averages their absolute Pearson correlations.

In [ ]:
WITHIN_K = min(3, N_COMPONENTS)
within_models = {name: pd.concat([pd.read_csv(ANALYSIS_ROOT / pairing / f'{name}.csv').assign(pairing=pairing)
                                for pairing in PAIRINGS], ignore_index=True)
                 for name in ['within_model_metrics', 'within_model_pairs', 'within_model_correlations']}
display(within_models['within_model_metrics'].query('k == @WITHIN_K'))

In [ ]:
result = within_models
k = WITHIN_K
partition = 'in_sample'

if within_models is not None:
    import matplotlib.pyplot as plt
    table = result['within_model_metrics']
    table = table[(table.k == k) & (table.partition == partition)]
    if table.empty:
        raise ValueError('No results for this dimension/partition.')
    modalities = [m for m in ('ieeg','meg') if m in set(table.modality)]
    labels = {'separate_pca':'PCA', 'plssvd':'PLSSVD', 'joint_pca':'Joint PCA'}
    columns = [(space,metric) for space in ('temporal_scores','spatial_patterns')
               for metric in ('overlap','matched_abs_r')]
    fig, axes = plt.subplots(1,len(modalities),figsize=(15,3.8),squeeze=False,layout='constrained')
    for ax, modality in zip(axes.flat,modalities):
        sub = table[table.modality == modality]
        pairs = list(sub[['model_a','model_b']].drop_duplicates().itertuples(index=False,name=None))
        matrix = np.full((len(pairs),len(columns)),np.nan)
        spread = matrix.copy()
        for i,(a,b) in enumerate(pairs):
            for j,(space,metric) in enumerate(columns):
                records = sub[(sub.model_a==a)&(sub.model_b==b)&(sub.space==space)]
                values = records.groupby('pairing')[metric].median().reindex(PAIRINGS).to_numpy()
                matrix[i,j], spread[i,j] = summarize(values)
        im = ax.imshow(np.ma.masked_invalid(matrix),vmin=0,vmax=1,cmap='viridis',aspect='auto')
        ax.set(title='iEEG' if modality=='ieeg' else 'MEG',
               xticks=range(4),xticklabels=['Time subspace\noverlap','Time components\nmean |r|','Spatial subspace\noverlap','Spatial components\nmean |r|'],
               yticks=range(len(pairs)),yticklabels=[labels.get(a,a)+' vs '+labels.get(b,b) for a,b in pairs])
        ax.axvline(1.5,color='white',lw=2)
        for i in range(len(pairs)):
            for j in range(4):
                v = matrix[i,j]
                ax.text(j,i,(f'{v:.2f}\n±{spread[i,j]:.2f}' if MODE == 'random' else f'{v:.2f}') if np.isfinite(v) else '—',ha='center',va='center',
                        color='black' if v>.55 else 'white',fontsize=11)
    fig.colorbar(im,ax=axes.ravel().tolist(),label='Similarity (0–1)',shrink=.8)
    setting = 'Same-data comparison' if partition=='in_sample' else partition
    fig.suptitle(f'Within-modality model similarity · {k} components · {setting}' + (f' · mean ± {UNCERTAINTY}' if MODE == 'random' else ''))
    fig.supxlabel('Equal condition averages · component pairs chosen from time courses · mean across pairings',fontsize=9)
    for ax in fig.axes: ax.grid(True)
    plt.show()

In [ ]:
result = within_models
k = N_COMPONENTS
partition = 'in_sample'

if within_models is not None:
    import matplotlib.pyplot as plt
    corr = result['within_model_correlations']
    corr = corr[(corr.k == k) & (corr.partition == partition)]
    if corr.empty:
        raise ValueError('No component correlations for this dimension/partition.')
    repeat = corr.repeat.min()
    corr = corr[corr.repeat == repeat].copy()
    pairs = result['within_model_pairs']
    pairs = pairs[(pairs.k == k) & (pairs.repeat == repeat) & pairs.valid_training_pair]
    keys = ['pairing','modality','model_a','model_b','k','repeat','component_a','component_b']
    matched = corr.merge(pairs[keys], on=keys, validate='many_to_one')
    if MODE == 'random':
        for side in ('a', 'b'):
            matched['component_'+side] = [ALIGNMENT[p][m][int(c)] for p,m,c in zip(matched.pairing, matched['model_'+side], matched['component_'+side])]
    matched['abs_r'] = abs(matched.signed_r)
    labels = {'separate_pca':'PCA','joint_pca':'Joint PCA','plssvd':'PLSSVD'}
    fig, axes = plt.subplots(2,2,figsize=(max(11,k*1.05),6),layout='constrained')
    for row,modality in enumerate(('ieeg','meg')):
        sub = corr[corr.modality == modality]
        model_pairs = list(sub[['model_a','model_b']].drop_duplicates().itertuples(index=False,name=None))
        for col,space in enumerate(('temporal_scores','spatial_patterns')):
            ax = axes[row,col]
            matrix = np.full((len(model_pairs),k),np.nan)
            annotations = {}
            for i,(a,b) in enumerate(model_pairs):
                records = matched[(matched.modality==modality)&(matched.model_a==a)&(matched.model_b==b)&(matched.space==space)]
                for component, group in records.groupby('component_a'):
                    j = int(component)-1
                    values = group.groupby('pairing').abs_r.mean().reindex(PAIRINGS).to_numpy()
                    mean, spread = summarize(values)
                    matrix[i,j] = mean
                    rank = int(group.component_b.mode().iloc[0])
                    annotations[i,j] = (f'{mean:.2f}±{spread:.2f}\n{component}→{rank} (mode)' if MODE == 'random'
                                        else f'{mean:.2f}\n{component}→{rank}')
            im = ax.imshow(np.ma.masked_invalid(matrix),cmap='viridis',vmin=0,vmax=1,aspect='auto')
            ax.set(title=f'{"iEEG" if modality=="ieeg" else "MEG"} · {"Time courses" if col==0 else "Spatial patterns"}',
                   xticks=range(k),xticklabels=range(1,k+1),xlabel='Component in first model (A)',
                   yticks=range(len(model_pairs)),yticklabels=[f'{labels.get(a,a)} vs {labels.get(b,b)}' for a,b in model_pairs])
            for i in range(len(model_pairs)):
                for j in range(k):
                    ax.text(j,i,annotations.get((i,j),'—'),ha='center',va='center',fontsize=8,
                            color='black' if matrix[i,j]>.55 else 'white')
    fig.colorbar(im,ax=axes.ravel().tolist(),label='Individual matched |Pearson r|',shrink=.8)
    fig.suptitle(f'Individual components · matching within first {k} · {partition} · {MODE}' + (f' · mean ± {UNCERTAINTY}' if MODE == 'random' else ''))
    fig.supxlabel('Each cell: matched |r| and component A→B; random: mean ± uncertainty, modal B rank after reference alignment. Spatial comparisons reuse temporal pairs.',fontsize=9)
    for ax in fig.axes: ax.grid(True)
    plt.show()